In [2]:
"""
Weighted Graph with Heuristic Values & A* Pathfinding
Nodes: A, B, C, D, E, G
Start: A  |  Goal: G
Draws the graph, highlights the optimal path, and saves as graph.png
"""

import heapq
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving only
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# ============================================================
# 1. GRAPH DEFINITION
# ============================================================
graph = nx.Graph()

# Nodes with heuristic values h(n) -- estimated cost to goal G
heuristics = {
    'A': 10,
    'B': 8,
    'C': 5,
    'D': 7,
    'E': 3,
    'G': 0,   # Goal node
}

START = 'A'
GOAL  = 'G'

for node, h in heuristics.items():
    graph.add_node(node, heuristic=h)

# Weighted edges (node1, node2, weight)
edges = [
    ('A', 'B', 1),
    ('A', 'C', 4),
    ('B', 'C', 2),
    ('B', 'D', 5),
    ('C', 'E', 3),
    ('D', 'E', 1),
    ('D', 'G', 6),
    ('E', 'G', 2),
]

for u, v, w in edges:
    graph.add_edge(u, v, weight=w)

# ============================================================
# 2. A* PATHFINDING ALGORITHM (Step-by-Step)
# ============================================================
def astar(graph, start, goal, heuristics):
    """
    A* search algorithm.
    f(n) = g(n) + h(n)
      g(n) = actual cost from start to n
      h(n) = heuristic estimate from n to goal
    Returns: (path, total_cost, exploration_log)
    """
    # Priority queue entries: (f_cost, tie_breaker, node, path, g_cost)
    counter = 0
    open_set = [(heuristics[start], counter, start, [start], 0)]
    visited = set()
    log = []  # Step-by-step exploration log

    while open_set:
        f, _, current, path, g = heapq.heappop(open_set)

        if current in visited:
            continue

        log.append({
            'node': current,
            'g': g,
            'h': heuristics[current],
            'f': f,
            'path': list(path),
        })

        if current == goal:
            return path, g, log

        visited.add(current)

        for neighbor in graph.neighbors(current):
            if neighbor not in visited:
                edge_w = graph[current][neighbor]['weight']
                new_g = g + edge_w
                new_f = new_g + heuristics[neighbor]
                counter += 1
                heapq.heappush(open_set, (new_f, counter, neighbor, path + [neighbor], new_g))

    return None, float('inf'), log  # No path found


optimal_path, total_cost, exploration_log = astar(graph, START, GOAL, heuristics)

# Build set of path edges for highlighting
path_edges = set()
if optimal_path:
    for i in range(len(optimal_path) - 1):
        path_edges.add((optimal_path[i], optimal_path[i + 1]))
        path_edges.add((optimal_path[i + 1], optimal_path[i]))  # undirected

# ============================================================
# 3. DRAW THE GRAPH
# ============================================================
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor('#1e1e2f')
ax.set_facecolor('#1e1e2f')

pos = nx.spring_layout(graph, seed=42, k=2.0)

# --- Edges: dim non-path edges, highlight path edges ---
non_path_edges = [(u, v) for u, v in graph.edges() if (u, v) not in path_edges]
path_edge_list = [(u, v) for u, v in graph.edges() if (u, v) in path_edges]

nx.draw_networkx_edges(
    graph, pos, ax=ax,
    edgelist=non_path_edges,
    width=2.0, edge_color='#565f89', alpha=0.5, style='dashed',
)
nx.draw_networkx_edges(
    graph, pos, ax=ax,
    edgelist=path_edge_list,
    width=4.0, edge_color='#ff9e64', alpha=1.0,
)

# --- Edge weight labels ---
edge_labels = nx.get_edge_attributes(graph, 'weight')
nx.draw_networkx_edge_labels(
    graph, pos, ax=ax,
    edge_labels=edge_labels,
    font_size=13, font_color='#e0af68', font_weight='bold',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='#24283b', edgecolor='#e0af68', alpha=0.9),
)

# --- Nodes: color-code Start, Goal, Path, and Others ---
def get_node_color(n):
    if n == START:
        return '#bb9af7'   # Purple -- Start
    if n == GOAL:
        return '#9ece6a'   # Green  -- Goal
    if optimal_path and n in optimal_path:
        return '#ff9e64'   # Orange -- On optimal path
    return '#7dcfff'       # Cyan   -- Regular

node_colors = [get_node_color(n) for n in graph.nodes()]

nx.draw_networkx_nodes(
    graph, pos, ax=ax,
    node_size=1000,
    node_color=node_colors,
    edgecolors='#c0caf5', linewidths=2.5,
)

# --- Node labels: show name + h(n) ---
node_labels = {n: f"{n}\nh={heuristics[n]}" for n in graph.nodes()}
nx.draw_networkx_labels(
    graph, pos, ax=ax,
    labels=node_labels,
    font_size=11, font_color='#1a1b26', font_weight='bold',
)

# --- Title ---
ax.set_title(
    f'A* Pathfinding :  {START} -> {GOAL}   |   Optimal Cost = {total_cost}',
    fontsize=17, fontweight='bold', color='#c0caf5', pad=20,
)

# --- Legend ---
legend_elements = [
    mpatches.Patch(facecolor='#bb9af7', edgecolor='#c0caf5', label=f'Start Node ({START})'),
    mpatches.Patch(facecolor='#9ece6a', edgecolor='#c0caf5', label=f'Goal Node ({GOAL})'),
    mpatches.Patch(facecolor='#ff9e64', edgecolor='#c0caf5', label='Optimal Path Node'),
    mpatches.Patch(facecolor='#7dcfff', edgecolor='#c0caf5', label='Other Node'),
    mpatches.Patch(facecolor='#24283b', edgecolor='#e0af68', label='Edge Weight'),
]
ax.legend(
    handles=legend_elements, loc='lower left',
    fontsize=9, facecolor='#24283b', edgecolor='#565f89',
    labelcolor='#c0caf5',
)

ax.axis('off')
plt.tight_layout()
plt.savefig('graph.png', dpi=150, facecolor='#1e1e2f', bbox_inches='tight')
print("[OK] Graph saved as graph.png")

# ============================================================
# 4. A* STEP-BY-STEP NODE EXPANSION (Console Output)
# ============================================================
print("")
print("=" * 65)
print("   A* ALGORITHM  --  STEP-BY-STEP NODE EXPANSION")
print("=" * 65)

print("")
print(f"  Start Node : {START}   (h = {heuristics[START]})")
print(f"  Goal Node  : {GOAL}    (h = {heuristics[GOAL]})")
print(f"  Evaluation : f(n) = g(n) + h(n)")
print(f"    g(n) = actual cost from {START} to n")
print(f"    h(n) = heuristic estimate from n to {GOAL}")

print("")
print("-" * 65)
print("  GRAPH STRUCTURE")
print("-" * 65)
print(f"  Nodes: {list(graph.nodes())}")
print(f"  Edges & Weights:")
for u, v, d in graph.edges(data=True):
    print(f"    {u} --({d['weight']})-- {v}")
print(f"  Heuristic Values h(n):")
for n, h in heuristics.items():
    print(f"    h({n}) = {h}")

print("")
print("-" * 65)
print("  A* EXPLORATION STEPS")
print("-" * 65)
print(f"  {'Step':<6} {'Expand':<8} {'g(n)':<7} {'h(n)':<7} {'f(n)':<7} {'Path So Far'}")
print(f"  {'----':<6} {'------':<8} {'----':<7} {'----':<7} {'----':<7} {'-'*20}")

for i, step in enumerate(exploration_log, 1):
    path_str = ' -> '.join(step['path'])
    print(f"  {i:<6} {step['node']:<8} {step['g']:<7} {step['h']:<7} {step['f']:<7} {path_str}")

    # Show neighbors being added to the open set
    if step['node'] != GOAL:
        visited_so_far = set(s['node'] for s in exploration_log[:i])
        neighbors_info = []
        for nb in graph.neighbors(step['node']):
            if nb not in visited_so_far or nb == step['node']:
                edge_w = graph[step['node']][nb]['weight']
                nb_g = step['g'] + edge_w
                nb_f = nb_g + heuristics[nb]
                neighbors_info.append((nb, edge_w, nb_g, heuristics[nb], nb_f))
        if neighbors_info:
            print(f"         -> Neighbors added to open set:")
            for nb, ew, ng, nh, nf in neighbors_info:
                print(f"            {step['node']}->{nb}: g={ng} (={step['g']}+{ew}), h={nh}, f={nf}")
    print("")

print("-" * 65)
print("  RESULT")
print("-" * 65)
if optimal_path:
    print(f"  Optimal Path : {' -> '.join(optimal_path)}")
    print(f"  Total Cost   : {total_cost}")
    print("")
    print(f"  Path Breakdown:")
    running = 0
    for i in range(len(optimal_path) - 1):
        u, v = optimal_path[i], optimal_path[i + 1]
        w = graph[u][v]['weight']
        running += w
        print(f"    {u} -> {v}  (edge cost = {w}, cumulative = {running})")
    print(f"    {'-'*40}")
    print(f"    Total path cost = {total_cost}")
else:
    print(f"  [FAIL] No path found from {START} to {GOAL}")

print("")
print("=" * 65)


[OK] Graph saved as graph.png

   A* ALGORITHM  --  STEP-BY-STEP NODE EXPANSION

  Start Node : A   (h = 10)
  Goal Node  : G    (h = 0)
  Evaluation : f(n) = g(n) + h(n)
    g(n) = actual cost from A to n
    h(n) = heuristic estimate from n to G

-----------------------------------------------------------------
  GRAPH STRUCTURE
-----------------------------------------------------------------
  Nodes: ['A', 'B', 'C', 'D', 'E', 'G']
  Edges & Weights:
    A --(1)-- B
    A --(4)-- C
    B --(2)-- C
    B --(5)-- D
    C --(3)-- E
    D --(1)-- E
    D --(6)-- G
    E --(2)-- G
  Heuristic Values h(n):
    h(A) = 10
    h(B) = 8
    h(C) = 5
    h(D) = 7
    h(E) = 3
    h(G) = 0

-----------------------------------------------------------------
  A* EXPLORATION STEPS
-----------------------------------------------------------------
  Step   Expand   g(n)    h(n)    f(n)    Path So Far
  ----   ------   ----    ----    ----    --------------------
  1      A        0       10      10 